# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

### Set-up

In [9]:
%load_ext autoreload
%autoreload 2    

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:

import arviz as az

In [17]:
import sys
from pathlib import Path
root = Path.cwd().parent
sys.path.insert(0, str(root))

import part1  # we will reuse some things from last tutorial
import part2

In [19]:
# save models and outputs there to keep the directory tree organised
results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True) 
results_folder

WindowsPath('results')

### Data-loading
re-using the code from the first tutorial to load and prepare our training and validation data

In [20]:
from datasets import load_dataset

agnews_outcome_space = part1.clslib.NamedClasses(['world', 'sports', 'business', 'tech/sci-fi'])
print(agnews_outcome_space)

#Loading our training corpus
dataset = load_dataset("fancyzhx/ag_news") 
print(dataset)

corpus = dataset["train"]["text"] 

0: world, 1: sports, 2: business, 3: tech/sci-fi


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


As in this tutorial we study a number of designs, we use cross-validation to obtain more reliable estimates of their merits in classification.

In [21]:
%%time
my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

for seed in my_seeds:  
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(dataset["train"]["text"], dataset["train"]["label"])),  
            k=num_folds,
            seed=seed
        )
    )

CPU times: total: 2.05 s
Wall time: 2.03 s


We will not be using the test set `dataset["test"]` for most of this tutorial, but we already split it in a `devtest` set (for some analysis at the end) and its complement (as an unseen dataset).

In [22]:
len(dataset["test"]["text"])

7600

In [23]:
my_test_sets = []
devtest_size = 100 * len(agnews_outcome_space)
for seed in my_seeds:
    my_test_sets.append(part1.experiment.split_test_devtest(zip(dataset["test"]["text"], dataset["test"]["label"]), devtest_size, seed=seed))

In [24]:
my_test_sets[0].keys(), len(my_test_sets[0]['devtest']), len(my_test_sets[0]['test'])

(dict_keys(['devtest', 'devtest_ids', 'test', 'test_ids']), 400, 7200)

### Pre-trained embeddings

#### Glove

In [25]:
from pathlib import Path
from urllib.request import urlopen
import shutil

def download_pretrained(option):
    """
    You can use this to download GloVe (or similar) model using wget. 
    It's fine to download it through other means, but don't unzip the file (the rest of the notebook can 
    handle the file in its zipped format).

    This function returns the path which you can use for the loader class that follows.

    The function will skip the download if it detects the file in disk. If the file in disk is corrupted, you will have to delete the file first.
    """
    if option == 'glove':
        destination = "glove.2024.wikigiga.50d.zip"
        dims = 50
        skip_first = False
    elif option == 'fasttext':
        destination = "wiki-news-300d-1M.vec.zip"
        dims = 300
        skip_first = True
    else:
        raise ValueError(f"Unkonwn option: {option}")
        
    if not Path(destination).is_file():
        if option == 'glove':
            !wget -O glove.2024.wikigiga.50d.zip 'https://surfdrive.surf.nl/s/qYfZ6PiNrMLCwpS/download'
        elif option == 'fasttext':
            !wget https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
        

    return destination, dims, skip_first

Here we download the model:

In [26]:
%%time
# depends on internet connection (but say 5 minutes or less)
glv_embtable_version, glv_embtable_dims, glv_embtable_metadata = download_pretrained('glove') # in case stanford.edu is down, use fasttext (but glove is preferred)
glv_embtable_version, glv_embtable_dims, glv_embtable_metadata

CPU times: total: 0 ns
Wall time: 0 ns


('glove.2024.wikigiga.50d.zip', 50, False)

In `part2/clslib.py`, you will find `PretrainedEmbeddings`, which can be used to load embeddings to memory. We've documented it well for you, in case you want to familiarise yourself with the relevant steps.

In [43]:
%%time
# about 30 seconds for GloVe
glv_trained_emb_model = part2.clslib.PretrainedEmbeddings(glv_embtable_version, glv_embtable_dims, skip_first_line=glv_embtable_metadata)

CPU times: total: 33.2 s
Wall time: 33.4 s


Internally, this object maintains two important things:
- a Vocabulary object that manages the correspondence between words and their 0-based unique ids
- an embedding matrix, whose rows align with the correspondence established by the vocabulary

In [28]:
print("Shape of embedding matrix:", glv_trained_emb_model.vectors.shape)
print("Size of vocabulary:", len(glv_trained_emb_model.vocab))
print(f"UNK token {glv_trained_emb_model.vocab.default_index}:{glv_trained_emb_model.vocab.default_word}") 
print(f"PAD token {glv_trained_emb_model.vocab.index(glv_trained_emb_model.pad_token)}:{glv_trained_emb_model.pad_token}")
print("First few entries in the vocabulary:")
for _, (i, w) in zip(range(10), glv_trained_emb_model.vocab.items()):
    print(f"{i}:{w}")

Shape of embedding matrix: (1287615, 50)
Size of vocabulary: 1287615
UNK token 1:<unk>
PAD token 0:<pad>
First few entries in the vocabulary:
0:<pad>
1:<unk>
2:the
3:,
4:.
5:of
6:to
7:and
8:in
9:a


#### word2vec

In [29]:
import gensim.downloader as api

word2vec_model = api.load("word2vec-google-news-300")

In [30]:
print("Vocabulary size:", len(word2vec_model))
print("Embedding dimension:", word2vec_model.vector_size)

print(word2vec_model["computer"].shape)

Vocabulary size: 3000000
Embedding dimension: 300
(300,)


adapt the form of word2vec

In [49]:
import numpy as np

import part2.clslib

class GensimPretrainedEmbeddings(part2.clslib.PretrainedEmbeddings):

    def __init__(self, keyed_vectors, keep_words=None):
        self.num_dims = keyed_vectors.vector_size
        self.pad_token = "<pad>"
        self.unk_token = "<unk>"

        self.vocab = part2.clslib.Vocabulary(
            init=[self.pad_token, self.unk_token],
            default=self.unk_token
        )

        vectors = [
            np.zeros(self.num_dims, dtype=np.float32),
            np.zeros(self.num_dims, dtype=np.float32)
        ]

        # Optional: keep only words needed for the dataset(for this project, using word in AG news for downstream classfication)
        if keep_words is not None:
            keep_words = {w.lower() for w in keep_words}

        for word in keyed_vectors.index_to_key:
            word_lower = word.lower()

            if keep_words is not None and word_lower not in keep_words:
                continue

            if word_lower in self.vocab:
                continue

            self.vocab.add(word_lower)
            vectors.append(
                np.asarray(keyed_vectors[word], dtype=np.float32)
            )

        self.vectors = np.stack(vectors, axis=0)

        # Use mean embedding for unknown words
        if len(vectors) > 2:
            self.vectors[1] = self.vectors[2:].mean(axis=0)

        self.vocab.freeze()

        assert len(self.vocab) == self.vectors.shape[0]

In [50]:
glove = glv_trained_emb_model
word2vec = GensimPretrainedEmbeddings(word2vec_model)

### Tokenizer

In [34]:
class Tokenizer:
    def tokens(self, text: str) -> list:
        """Tokenize as a list of string tokens"""
        raise NotImplementedError("Implement me in subclass")
    def ids(self, text: str) -> list:
        """Tokenize as a list of token ids"""
        raise NotImplementedError("Implement me in subclass")

In [35]:
import nltk

nltk.download('stopwords')
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\heyez\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\heyez\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [37]:
from tabulate import tabulate

demo_texts = ['Dog bites man.', 'This tutorial is offered as part of NLP1, an MSc level course in the AI programme of the UvA.']



A compositional encoder on the other hand can make use of everything that's in the sequence, so we keep punctuation and stopwords:

In [41]:
glove_tokenizer = part2.clslib.BasicTokenizer(
    lowercase=True,
    remove_punct=True,
    remove_stopwords=False,
    vocab=glove.vocab
)

for text in demo_texts:
    print(text)
    print(tabulate(list(zip(glove_tokenizer.ids(text), glove_tokenizer.tokens(text))), headers=['id', 'token']))    
    print()


Dog bites man.
   id  token
-----  -------
 3039  dog
16342  bites
  311  man

This tutorial is offered as part of NLP1, an MSc level course in the AI programme of the UvA.
   id  token
-----  ---------
   41  this
42158  tutorial
   17  is
 1123  offered
   21  as
  157  part
    5  of
    1  nlp1
   31  an
24596  msc
  502  level
  760  course
    8  in
    2  the
 7388  ai
 3024  programme
    5  of
    2  the
45065  uva



In [51]:
word2vec_tokenizer = part2.clslib.BasicTokenizer(
    lowercase=False,
    remove_punct=True,
    remove_stopwords=False,
    vocab= word2vec.vocab
)

for text in demo_texts:
    print(text)
    print(tabulate(list(zip(word2vec_tokenizer.ids(text), word2vec_tokenizer.tokens(text))), headers=['id', 'token']))    
    print()

Dog bites man.
   id  token
-----  -------
    1  Dog
16803  bites
  237  man

This tutorial is offered as part of NLP1, an MSc level course in the AI programme of the UvA.
   id  token
-----  ---------
    1  This
27023  tutorial
    6  is
  957  offered
   15  as
  176  part
 1663  of
    1  NLP1
   28  an
    1  MSc
  393  level
  665  course
    3  in
    9  the
    1  AI
11435  programme
 1663  of
    9  the
    1  UvA



In [53]:
sgns_tokenizer = part2.clslib.BasicTokenizer(
    lowercase=False,
    remove_punct=True,
    remove_stopwords=False,
)

for text in demo_texts:
    print(text)
    print(sgns_tokenizer.tokens(text))    
    print()

Dog bites man.
['Dog', 'bites', 'man']

This tutorial is offered as part of NLP1, an MSc level course in the AI programme of the UvA.
['This', 'tutorial', 'is', 'offered', 'as', 'part', 'of', 'NLP1', 'an', 'MSc', 'level', 'course', 'in', 'the', 'AI', 'programme', 'of', 'the', 'UvA']



#### training in-domain SGNS embedding model

In [54]:
sgns_tokenized_corpus = [
    sgns_tokenizer.tokens(text)
    for text in corpus
]

print("Number of documents:", len(sgns_tokenized_corpus))
print("First document:", sgns_tokenized_corpus[0])

Number of documents: 120000
First document: ['Wall', 'St', 'Bears', 'Claw', 'Back', 'Into', 'the', 'Black', 'Reuters', 'Reuters', 'Short', 'sellers', 'Wall', 'Street', 's', 'dwindling', 'band', 'of', 'ultra', 'cynics', 'are', 'seeing', 'green', 'again']


In [59]:
from pathlib import Path
from gensim.models import Word2Vec

model_path = Path("models/agnews_sgns_50d.model")
model_path.parent.mkdir(parents=True, exist_ok=True)

if model_path.exists():
    print("Loading existing SGNS model...")
    sgns_model = Word2Vec.load(str(model_path))
else:
    print("Training new SGNS model...")

    sgns_model = Word2Vec(
        sentences=sgns_tokenized_corpus,
        vector_size=50,
        window=5,
        min_count=2,
        sg=1,
        hs=0,
        negative=5,
        epochs=10,
        workers=1,
        seed=42
    )

    sgns_model.save(str(model_path))
    print("Model saved!")

Loading existing SGNS model...


In [56]:
print("Vocabulary size:", len(sgns_model.wv))
print("Embedding dimension:", sgns_model.wv.vector_size)
print("Embedding matrix shape:", sgns_model.wv.vectors.shape)

Vocabulary size: 57768
Embedding dimension: 50
Embedding matrix shape: (57768, 50)


adapt sgns 

In [61]:
sgns = GensimPretrainedEmbeddings(sgns_model.wv)

test a set of word in all emb models

In [62]:
models = {
    "GloVe": glove,
    "Word2Vec": word2vec,
    "In-domain SGNS": sgns
}

words = [
    "government", "election",
    "football", "championship",
    "stock", "market",
    "software", "technology"
] #using words related to news

for name, model in models.items():
    print(f"\n{name}")

    for word in words:
        if word in model:
            print(
                word,
                model.index(word),
                model.embed_word(word).shape
            )
        else:
            print(word, "OOV")


GloVe
government 80 (50,)
election 375 (50,)
football 552 (50,)
championship 783 (50,)
stock 479 (50,)
market 223 (50,)
software 1649 (50,)
technology 767 (50,)

Word2Vec
government 116 (300,)
election 595 (300,)
football 679 (300,)
championship 1188 (300,)
stock 526 (300,)
market 167 (300,)
software 798 (300,)
technology 425 (300,)

In-domain SGNS
government 94 (50,)
election 229 (50,)
football 516 (50,)
championship 655 (50,)
stock 358 (50,)
market 129 (50,)
software 115 (50,)
technology 179 (50,)
